In [8]:
import os
import re
import time
from pathlib import Path

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_DIR = Path("/home/eleroy/proj-dirs/fast-o3-iv/data/external/NCAR/CESM2/ssp245")

VARIABLE = "PS"

filelist = [
    f"https://osdf-director.osg-htc.org/ncar/gdex/d651073/b.e21.BSSP245smbb.f09_g17/b.e21.BSSP245smbb.f09_g17.001/atm/proc/tseries/month_1/b.e21.BSSP245smbb.f09_g17.001.cam.h0.{VARIABLE}.201501-206412.nc",
    f"https://osdf-director.osg-htc.org/ncar/gdex/d651073/b.e21.BSSP245smbb.f09_g17/b.e21.BSSP245smbb.f09_g17.002/atm/proc/tseries/month_1/b.e21.BSSP245smbb.f09_g17.002.cam.h0.{VARIABLE}.201501-206412.nc",
    f"https://osdf-director.osg-htc.org/ncar/gdex/d651073/b.e21.BSSP245smbb.f09_g17/b.e21.BSSP245smbb.f09_g17.003/atm/proc/tseries/month_1/b.e21.BSSP245smbb.f09_g17.003.cam.h0.{VARIABLE}.201501-206412.nc",
    f"https://osdf-director.osg-htc.org/ncar/gdex/d651073/b.e21.BSSP245smbb.f09_g17/b.e21.BSSP245smbb.f09_g17.004/atm/proc/tseries/month_1/b.e21.BSSP245smbb.f09_g17.004.cam.h0.{VARIABLE}.201501-206412.nc",
    f"https://osdf-director.osg-htc.org/ncar/gdex/d651073/b.e21.BSSP245smbb.f09_g17/b.e21.BSSP245smbb.f09_g17.005/atm/proc/tseries/month_1/b.e21.BSSP245smbb.f09_g17.005.cam.h0.{VARIABLE}.201501-206412.nc",
    f"https://osdf-director.osg-htc.org/ncar/gdex/d651073/b.e21.BSSP245smbb.f09_g17/b.e21.BSSP245smbb.f09_g17.006/atm/proc/tseries/month_1/b.e21.BSSP245smbb.f09_g17.006.cam.h0.{VARIABLE}.201501-206412.nc",
    f"https://osdf-director.osg-htc.org/ncar/gdex/d651073/b.e21.BSSP245smbb.f09_g17/b.e21.BSSP245smbb.f09_g17.007/atm/proc/tseries/month_1/b.e21.BSSP245smbb.f09_g17.007.cam.h0.{VARIABLE}.201501-206412.nc",
    f"https://osdf-director.osg-htc.org/ncar/gdex/d651073/b.e21.BSSP245smbb.f09_g17/b.e21.BSSP245smbb.f09_g17.008/atm/proc/tseries/month_1/b.e21.BSSP245smbb.f09_g17.008.cam.h0.{VARIABLE}.201501-206412.nc",
    f"https://osdf-director.osg-htc.org/ncar/gdex/d651073/b.e21.BSSP245smbb.f09_g17/b.e21.BSSP245smbb.f09_g17.009/atm/proc/tseries/month_1/b.e21.BSSP245smbb.f09_g17.009.cam.h0.{VARIABLE}.201501-206412.nc",
    f"https://osdf-director.osg-htc.org/ncar/gdex/d651073/b.e21.BSSP245smbb.f09_g17/b.e21.BSSP245smbb.f09_g17.010/atm/proc/tseries/month_1/b.e21.BSSP245smbb.f09_g17.010.cam.h0.{VARIABLE}.201501-206412.nc",
    f"https://osdf-director.osg-htc.org/ncar/gdex/d651073/b.e21.BSSP245smbb.f09_g17/b.e21.BSSP245smbb.f09_g17.011/atm/proc/tseries/month_1/b.e21.BSSP245smbb.f09_g17.011.cam.h0.{VARIABLE}.201501-206412.nc",
    f"https://osdf-director.osg-htc.org/ncar/gdex/d651073/b.e21.BSSP245smbb.f09_g17/b.e21.BSSP245smbb.f09_g17.012/atm/proc/tseries/month_1/b.e21.BSSP245smbb.f09_g17.012.cam.h0.{VARIABLE}.201501-206412.nc",
    f"https://osdf-director.osg-htc.org/ncar/gdex/d651073/b.e21.BSSP245smbb.f09_g17/b.e21.BSSP245smbb.f09_g17.013/atm/proc/tseries/month_1/b.e21.BSSP245smbb.f09_g17.013.cam.h0.{VARIABLE}.201501-206412.nc",
    f"https://osdf-director.osg-htc.org/ncar/gdex/d651073/b.e21.BSSP245smbb.f09_g17/b.e21.BSSP245smbb.f09_g17.014/atm/proc/tseries/month_1/b.e21.BSSP245smbb.f09_g17.014.cam.h0.{VARIABLE}.201501-206412.nc",
    f"https://osdf-director.osg-htc.org/ncar/gdex/d651073/b.e21.BSSP245smbb.f09_g17/b.e21.BSSP245smbb.f09_g17.015/atm/proc/tseries/month_1/b.e21.BSSP245smbb.f09_g17.015.cam.h0.{VARIABLE}.201501-206412.nc",
    f"https://osdf-director.osg-htc.org/ncar/gdex/d651073/b.e21.BSSP245smbb.f09_g17/b.e21.BSSP245smbb.f09_g17.016/atm/proc/tseries/month_1/b.e21.BSSP245smbb.f09_g17.016.cam.h0.{VARIABLE}.201501-206412.nc",


]

session = requests.Session()
retry = Retry(
    total=8,
    connect=8,
    read=8,
    backoff_factor=2.0,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
)
session.mount("https://", HTTPAdapter(max_retries=retry))
session.headers.update({"User-Agent": "fast-o3-iv-downloader/1.0", "Accept-Encoding": "identity"})


def download_with_resume(url: str, out_path: Path, attempts: int = 8) -> None:
    tmp_path = out_path.with_suffix(out_path.suffix + ".part")

    for i in range(1, attempts + 1):
        try:
            start = tmp_path.stat().st_size if tmp_path.exists() else 0
            headers = {"Range": f"bytes={start}-"} if start > 0 else {}

            with session.get(
                url,
                stream=True,
                timeout=(30, 300),
                headers=headers,
                allow_redirects=True,
            ) as r:
                if start > 0 and r.status_code == 200:
                    tmp_path.unlink(missing_ok=True)  # server ignored Range, restart
                    start = 0

                r.raise_for_status()
                mode = "ab" if start > 0 else "wb"
                with open(tmp_path, mode) as f:
                    for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                        if chunk:
                            f.write(chunk)

            tmp_path.replace(out_path)
            return

        except Exception as e:
            print(f"Attempt {i}/{attempts} failed for {url}: {e}")
            if i == attempts:
                raise
            time.sleep(min(60, 2**i))


for url in filelist:
    fname = os.path.basename(url)
    m = re.search(r"f09_g17\.(\d{3})", url)
    if not m:
        raise ValueError(f"Could not parse member from URL: {url}")
    member = m.group(1)
    var = fname.split(".cam.h0.")[1].split(".")[0]

    out_dir = BASE_DIR / member / var
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / fname

    if out_path.exists() and out_path.stat().st_size > 0:
        print(f"Skipping existing: {out_path}")
        continue

    print(f"Downloading {url} -> {out_path}")
    download_with_resume(url, out_path)
    print("done")


done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done


In [ ]:
import xarray as xr 

base = "/home/eleroy/proj-dirs/fast-o3-iv/data/external/NCAR/CESM2/ssp245/"
ds = xr.open_dataset(base + "016/U/b.e21.BSSP245smbb.f09_g17.016.cam.h0.U.201501-206412.nc", engine='netcdf4')

In [ ]:
ds